# Phase 4: Customer Analysis

CustomerKey is used instead of customer names so published outputs do not expose personal information.

In [ ]:
from pathlib import Path
import pandas as pd

project_folder = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cleaned_file = project_folder / 'data' / 'processed' / 'cleaned_sales.csv'
sales = pd.read_csv(cleaned_file, parse_dates=['Order Date'])
print('Rows loaded:', len(sales))

## 1. Customer sales, gross profit, and repeat purchases

In [ ]:
customer_performance = (
    sales.groupby(['CustomerKey', 'Name'], as_index=False)
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Gross_Profit=('Gross Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
    .sort_values('Revenue', ascending=False)
)
customer_performance[['Revenue', 'Gross_Profit']] = customer_performance[['Revenue', 'Gross_Profit']].round(2)
repeat_customers = customer_performance[customer_performance['Orders'] > 1]
top_20_customers = customer_performance.head(20)

print('Total customers:', len(customer_performance))
print('Repeat customers:', len(repeat_customers))
display(top_20_customers)

## 2. Customers contributing the first 80% of revenue

In [ ]:
customer_performance['Cumulative Revenue %'] = (
    customer_performance['Revenue'].cumsum()
    / customer_performance['Revenue'].sum()
    * 100
)
previous_percent = customer_performance['Cumulative Revenue %'].shift(fill_value=0)
customers_contributing_80_percent = customer_performance[previous_percent < 80]

print('Customers contributing the first 80%:', len(customers_contributing_80_percent))
display(customers_contributing_80_percent.head(20))

## 3. Customer retention rate

Retention is measured using the last two complete calendar years. It is the percentage of customers from the earlier year who also purchased in the following year. The partial 2021 year is excluded.

In [ ]:
months_per_year = sales.groupby('Year')['Month'].nunique()
complete_years = months_per_year[months_per_year == 12].index.tolist()
retention_year = max(complete_years)
base_year = retention_year - 1

base_customers = set(sales.loc[sales['Year'] == base_year, 'CustomerKey'])
returning_customers = set(sales.loc[sales['Year'] == retention_year, 'CustomerKey'])
retained_customers = base_customers.intersection(returning_customers)
retention_rate = len(retained_customers) / len(base_customers) * 100

print(f'Retention period: {base_year} to {retention_year}')
print('Customers in base year:', len(base_customers))
print('Retained customers:', len(retained_customers))
print(f'Customer retention rate: {retention_rate:.2f}%')

## 4. RFM values and scores

The snapshot date is one day after the final order. Each RFM measure is divided into five relative groups. Lower recency is better, while higher frequency and monetary value are better.

In [ ]:
snapshot_date = sales['Order Date'].max() + pd.Timedelta(days=1)
customer_rfm = (
    sales.groupby(['CustomerKey', 'Name'], as_index=False)
    .agg(
        First_Purchase_Date=('Order Date', 'min'),
        Last_Purchase_Date=('Order Date', 'max'),
        Frequency=('Order Number', 'nunique'),
        Monetary=('Revenue USD', 'sum'),
        Gross_Profit=('Gross Profit USD', 'sum'),
    )
)
customer_rfm['Recency'] = (snapshot_date - customer_rfm['Last_Purchase_Date']).dt.days
customer_rfm['R_Score'] = pd.qcut(customer_rfm['Recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
customer_rfm['F_Score'] = pd.qcut(customer_rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
customer_rfm['M_Score'] = pd.qcut(customer_rfm['Monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
customer_rfm['RFM_Score'] = customer_rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)
customer_rfm[['Monetary', 'Gross_Profit']] = customer_rfm[['Monetary', 'Gross_Profit']].round(2)

## 5. Customer segments

The combined score provides a simple ordered segmentation for this project. These are relative analytical groups, not company-approved customer definitions.

In [ ]:
def assign_customer_segment(score):
    if score >= 13:
        return 'Champions'
    if score >= 10:
        return 'Loyal Customers'
    if score >= 7:
        return 'Potential Loyalists'
    if score >= 4:
        return 'At Risk'
    return 'Lost Customers'

customer_rfm['Customer Segment'] = customer_rfm['RFM_Score'].apply(assign_customer_segment)
segment_summary = (
    customer_rfm.groupby('Customer Segment', as_index=False)
    .agg(Customers=('CustomerKey', 'count'), Revenue=('Monetary', 'sum'), Gross_Profit=('Gross_Profit', 'sum'))
    .sort_values('Revenue', ascending=False)
)
segment_summary[['Revenue', 'Gross_Profit']] = segment_summary[['Revenue', 'Gross_Profit']].round(2)
display(segment_summary)

In [ ]:
output_file = project_folder / 'data' / 'processed' / 'customer_segments.csv'
customer_rfm.to_csv(output_file, index=False, date_format='%Y-%m-%d')
print('Saved:', output_file)